In [6]:
!pip install nilearn --quiet
!pip -q install boto3 pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.5 MB/s eta 0:00:00


In [7]:
import os
import time
import boto3
import numpy as np
import nibabel as nib

from scipy import sparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from google.colab import userdata

In [12]:
N_SUBJECTS = 500
MAX_DOWNLOAD_WORKERS = 8

BUCKET = "hcp-openaccess"
S3_PREFIX = "HCP/"

ATLAS_FILE = "Q1-Q6_RelatedParcellation210.CorticalAreas_dil_Final_Final_Areas_Group_Colors.32k_fs_LR.dlabel.nii"

OUTPUT_FILE = "hcp_roi_timeseries.dat"
SUBJECT_FILE = "hcp_subjects_500.txt"

TMP_DIR = "hcp_tmp"
os.makedirs(TMP_DIR, exist_ok=True)

In [ ]:
from botocore.config import Config

config = Config(
    max_pool_connections=20
)

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="us-east-1",
    config=config
)

In [9]:
AWS_ACCESS_KEY = userdata.get("HCP_AWS_ACCESS_KEY")
AWS_SECRET_KEY = userdata.get("HCP_AWS_SECRET_KEY")

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="us-east-1"
)

print("S3 connection initialized.")

S3 connection initialized.


In [13]:
subjects = []

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(
    Bucket=BUCKET,
    Prefix=S3_PREFIX,
    Delimiter="/"
):
    for prefix in page.get("CommonPrefixes", []):
        subject = prefix["Prefix"].split("/")[1]

        if subject.isdigit():
            subjects.append(subject)

subjects = sorted(subjects, key=int)[:N_SUBJECTS]

print(f"Found {len(subjects)} subjects.")
print(subjects[:10])

Found 500 subjects.
['100307', '100408', '101006', '101107', '101309', '101410', '101915', '102008', '102311', '102816']


In [15]:
atlas = nib.load(ATLAS_FILE)

label_axis = atlas.header.get_axis(0)
atlas_labels = atlas.get_fdata().astype(np.int32).squeeze()

label_table = label_axis.label[0]

roi_ids = np.array(
    [label_id for label_id in label_table.keys() if label_id > 0],
    dtype=np.int32
)

roi_ids.sort()

print(f"Number of atlas ROIs: {len(roi_ids)}")
assert len(roi_ids) == 360

roi_to_column = {
    roi_id: i
    for i, roi_id in enumerate(roi_ids)
}

roi_column = np.full(atlas_labels.shape, -1, dtype=np.int16)

for roi_id, column in roi_to_column.items():
    roi_column[atlas_labels == roi_id] = column

valid = roi_column >= 0
counts = np.bincount(
    roi_column[valid],
    minlength=len(roi_ids)
)

weights = 1.0 / counts[roi_column[valid]]

roi_assignment = sparse.csr_matrix(
    (
        weights,
        (
            np.where(valid)[0],
            roi_column[valid]
        )
    ),
    shape=(len(atlas_labels), len(roi_ids))
)

print("ROI assignment matrix:", roi_assignment.shape)

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


Number of atlas ROIs: 360
ROI assignment matrix: (59412, 360)


In [16]:
test_subject = subjects[0]

test_key = (
    f"HCP/{test_subject}/MNINonLinear/Results/"
    f"rfMRI_REST1_LR/"
    f"rfMRI_REST1_LR_Atlas_hp2000_clean.dtseries.nii"
)

test_file = os.path.join(TMP_DIR, "test.dtseries.nii")

s3.download_file(BUCKET, test_key, test_file)

test_img = nib.load(test_file)

bold_axis = test_img.header.get_axis(1)

cortex_indices = []

for name, slc, bm in bold_axis.iter_structures():
    if "CORTEX" in name:
        cortex_indices.extend(range(slc.start, slc.stop))

cortex_indices = np.asarray(cortex_indices)

print("BOLD shape:", test_img.shape)
print("Cortical grayordinates:", len(cortex_indices))

assert len(cortex_indices) == len(atlas_labels)

os.remove(test_file)

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


BOLD shape: (1200, 91282)
Cortical grayordinates: 59412


In [17]:
N_TIMEPOINTS = test_img.shape[0]
N_ROIS = len(roi_ids)

roi_timeseries = np.memmap(
    OUTPUT_FILE,
    dtype=np.float32,
    mode="w+",
    shape=(len(subjects), N_TIMEPOINTS, N_ROIS)
)

np.save(
    SUBJECT_FILE.replace(".txt", ".npy"),
    np.asarray(subjects)
)

print("Output shape:", roi_timeseries.shape)
print(
    "Expected storage:",
    roi_timeseries.nbytes / (1024**3),
    "GB"
)

Output shape: (500, 1200, 360)
Expected storage: 0.8046627044677734 GB


In [18]:
def download_subject(subject):

    key = (
        f"HCP/{subject}/MNINonLinear/Results/"
        f"rfMRI_REST1_LR/"
        f"rfMRI_REST1_LR_Atlas_hp2000_clean.dtseries.nii"
    )

    local_file = os.path.join(
        TMP_DIR,
        f"{subject}.dtseries.nii"
    )

    try:
        s3.download_file(
            BUCKET,
            key,
            local_file
        )

        return subject, local_file, None

    except Exception as e:
        if os.path.exists(local_file):
            os.remove(local_file)

        return subject, None, str(e)

In [20]:
start_time = time.perf_counter()

completed = 0
failed = []

with ThreadPoolExecutor(
    max_workers=MAX_DOWNLOAD_WORKERS
) as executor:

    futures = {
        executor.submit(download_subject, subject): subject
        for subject in subjects
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Downloading and processing"
    ):

        subject, local_file, error = future.result()

        if error is not None:
            failed.append((subject, error))
            continue

        try:
            img = nib.load(local_file)

            data = img.get_fdata(
                dtype=np.float32
            )
            cortical_bold = data[:, cortex_indices]

            roi_ts = (
                cortical_bold @ roi_assignment
            )

            roi_timeseries[
                subjects.index(subject)
            ] = roi_ts

            completed += 1

        except Exception as e:

            failed.append(
                (subject, str(e))
            )

        finally:

            if os.path.exists(local_file):
                os.remove(local_file)


roi_timeseries.flush()

elapsed = time.perf_counter() - start_time

pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-

In [21]:
print("\n" + "=" * 50)

print(f"Completed: {completed}/{len(subjects)}")
print(f"Failed:    {len(failed)}")

print(f"\nTotal time: {elapsed / 60:.2f} minutes")

if completed > 0:
    print(
        f"Average time per subject: "
        f"{elapsed / completed:.2f} seconds"
    )

print(
    f"\nROI data shape: "
    f"{roi_timeseries.shape}"
)

print(
    f"ROI data size: "
    f"{roi_timeseries.nbytes / (1024**3):.2f} GB"
)

if failed:
    print("\nFailed subjects:")
    print(failed[:20])


Completed: 467/500
Failed:    33

Total time: 38.05 minutes
Average time per subject: 4.89 seconds

ROI data shape: (500, 1200, 360)
ROI data size: 0.80 GB

Failed subjects:
[('104012', 'An error occurred (404) when calling the HeadObject operation: Not Found'), ('105923', 'An error occurred (404) when calling the HeadObject operation: Not Found'), ('111514', 'An error occurred (404) when calling the HeadObject operation: Not Found'), ('116120', 'An error occurred (404) when calling the HeadObject operation: Not Found'), ('119833', 'could not broadcast input array from shape (1055,360) into shape (1200,360)'), ('126931', 'An error occurred (404) when calling the HeadObject operation: Not Found'), ('129432', 'An error occurred (404) when calling the HeadObject operation: Not Found'), ('131621', 'An error occurred (404) when calling the HeadObject operation: Not Found'), ('140420', 'could not broadcast input array from shape (1124,360) into shape (1200,360)'), ('143527', 'An error occur

In [43]:
from google.colab import drive
import os
import numpy as np

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/HCP_project"
os.makedirs(save_dir, exist_ok=True)

# Save ROI time series
np.save(
    os.path.join(save_dir, "roi_timeseries_500.npy"),
    roi_timeseries
)

# Save participant IDs
np.save(
    os.path.join(save_dir, "subjects_500.npy"),
    np.array(subjects)
)

print("Saved successfully.")
print("ROI data:", roi_timeseries.shape)
print("Location:", save_dir)

Mounted at /content/drive
Saved successfully.
ROI data: (500, 1200, 360)
Location: /content/drive/MyDrive/HCP_project


In [47]:
failed_ids = {subject for subject, error in failed}

keep_mask = np.array(
    [subject not in failed_ids for subject in subjects]
)

roi_timeseries_clean = roi_timeseries[keep_mask]
subjects_clean = np.asarray(subjects)[keep_mask]

In [48]:
np.save(
    os.path.join(save_dir, "roi_timeseries_467.npy"),
    roi_timeseries_clean
)

np.save(
    os.path.join(save_dir, "subjects_467.npy"),
    np.array(subjects_clean)
)